# Importation

In [46]:
import io
import os
import re
import shutil
import tarfile
import string
import torch
import tiktoken
import requests
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, TensorDataset
from torchinfo import summary
from sklearn.metrics import classification_report, confusion_matrix

# import matplotlib.pyplot as plt
import plotly.express as px
# from statsmodels.distributions.empirical_distribution import ECDF

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


Using cuda device


In [47]:
label_dic = {
    'spam' : 1,
    'ham' : 0
}

data = pd.read_csv('spam.csv', encoding='latin1')
data.drop(columns=data.columns[data.columns.str.contains('^Unnamed')], inplace=True)
data.columns = data.columns.str.strip()
data = data.rename(columns={'v1': 'labels', 'v2': 'text'})
data['labels'] = data['labels'].map(label_dic)
data

,labels,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will Ì_ b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


In [48]:
tokenizer = tiktoken.get_encoding("cl100k_base")

def encode_texts(texts):
    return [tokenizer.encode(text) for text in texts]

train_tokens = encode_texts(data["text"])

In [49]:
train_tokens
seq_lens = [len(seq) for seq in train_tokens]

fig = px.ecdf(data, x=seq_lens, title="nombre de tokens par phrase")
fig.show()
#Padding at 50
max_length = 50


In [50]:
def pad_sequences(sequences, max_length=max_length):
    return [seq[:max_length] + [0] * (max_length - len(seq)) for seq in sequences]

train_tokens = torch.tensor(pad_sequences(train_tokens), dtype=torch.long)
labels = torch.tensor(data['labels'], dtype=torch.bool)



In [51]:
# Define a custom PyTorch dataset class for ATT data
class SpamDataset(Dataset):
    """
    A custom dataset class for ATT data.

    This class is used to convert text data (already tokenized) and their corresponding labels
    into a PyTorch Dataset object, which can be easily loaded into a DataLoader.
    """

    def __init__(self, texts, labels):
        """
        Initializes the dataset by storing texts and labels as PyTorch tensors.

        Args:
        - texts (list or numpy array): Tokenized text data, where each text has been converted 
                                       into a sequence of word indices (integer tokens).
        - labels (list or numpy array): The corresponding labels for each text (e.g., sentiment scores or star ratings).
        """
        # Convert text sequences to a PyTorch tensor (long type since they are indices)
        self.texts = torch.tensor(texts, dtype=torch.long)

        # Convert labels to a PyTorch tensor (float32 for compatibility with loss functions)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        """
        Returns the total number of samples in the dataset.

        This method is required for PyTorch datasets as it allows DataLoader to determine
        how many batches it needs.
        """
        return len(self.texts)

    def __getitem__(self, idx):
        """
        Retrieves a single data point (text and label) from the dataset based on an index.

        Args:
        - idx (int): Index of the sample to retrieve.

        Returns:
        - tuple: A tuple containing:
            - self.texts[idx]: The tokenized text at index `idx`.
            - self.labels[idx]: The corresponding label for that text.
        """
        return self.texts[idx], self.labels[idx]

In [52]:
df_dataset = SpamDataset(train_tokens, labels)

# Split dataset into training (70%) and validation (15%) and test (15%)
train_size = int(0.7 * len(df_dataset))
val_size = len(df_dataset) - train_size

train_dataset, val_dataset = random_split(df_dataset, [train_size, val_size])
# val_dataset, test_dataset = random_split(temp_dataset, [val_size, test_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

C:\Users\dubos\AppData\Local\Temp\ipykernel_35756\3788339438.py:20: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).

C:\Users\dubos\AppData\Local\Temp\ipykernel_35756\3788339438.py:23: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).



In [53]:
text, label = next(iter(train_loader))
print(text)
print(label)

tensor([[   43,   337,   430,   596,  2204,    13,   358,  1541,   956,   733,
          4560,   311,  1505,  1475,  1972,  2324,  6685,   499,  3596,  3952,
            13,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [26158,   709,   311,   841,  3245,  7185,    13, 14910,   499,   617,
           264,  1695, 15553,    30,  3277,   527,   577,  3189, 10789, 71552,
            30,   358,  3940, 48986,  3432,    13,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [25821,   602,   617,  2834,  3243,   287,  1396,   220, 26491,   289,
           354,   656,   602,   656,  1828,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,

transfer Learning: 
- Which model to train on for spam ?
- Log training on mlflow ?
- Retrieve model from MLFlow

Metric validation:
- Unbalanced dataset => Accuracy out of the window. f1 score ?

In [54]:
# Get the vocabulary size from the tokenizer
# This represents the total number of unique words in the dataset,
# which will be used as the input size for the embedding layer.
vocab_size = tokenizer.n_vocab

# Define a neural network model for text classification
class SpamClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()

        # Embedding layer: Maps word indices to dense vector representations
        # padding_idx=0 ensures that padding tokens (index 0) do not contribute to learning
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Adaptive Average Pooling: Computes the average of the word embeddings along the sequence length
        # This helps reduce variable-length text into a fixed-size representation
        self.pooling = nn.AdaptiveAvgPool1d(1)

        # Fully Connected (Linear) layer: Maps the fixed-size vector to a single output value
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, text):
        # Convert input word indices into dense embeddings
        embedded = self.embedding(text)

        # Permute to match the expected shape for pooling: (batch, channels, sequence_length)
        # Then, apply average pooling to reduce sequence length to 1
        pooled = self.pooling(embedded.permute(0, 2, 1)).squeeze(2)

        # Pass the pooled embeddings through the linear layer to get the final prediction
        # return torch.sigmoid(self.fc(pooled))
        return self.fc(pooled)

# Create an instance of the model
model = SpamClassifier(vocab_size=vocab_size, embed_dim=16)

In [55]:
print(model)

# Print model summary
summary(model, input_data=text)  # (batch_size, input_features)



SpamClassifier(
  (embedding): Embedding(100277, 16, padding_idx=0)
  (pooling): AdaptiveAvgPool1d(output_size=1)
  (fc): Linear(in_features=16, out_features=1, bias=True)
)


Layer (type:depth-idx)                   Output Shape              Param #
SpamClassifier                           [16, 1]                   --
├─Embedding: 1-1                         [16, 50, 16]              1,604,432
├─AdaptiveAvgPool1d: 1-2                 [16, 16, 1]               --
├─Linear: 1-3                            [16, 1]                   17
Total params: 1,604,449
Trainable params: 1,604,449
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 25.67
Input size (MB): 0.01
Forward/backward pass size (MB): 0.10
Params size (MB): 6.42
Estimated Total Size (MB): 6.53

In [56]:
pos_weight = torch.tensor([4825/747])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train(model, train_loader, val_loader, criterion, optimizer, epochs=100):
    """
    Function to train a PyTorch model with training and validation datasets.
    
    Parameters:
    model: The neural network model to train.
    train_loader: DataLoader for the training dataset.
    val_loader: DataLoader for the validation dataset.
    criterion: Loss function (e.g., Binary Cross Entropy for classification).
    optimizer: Optimization algorithm (e.g., Adam, SGD).
    epochs: Number of training epochs (default=100).
    
    Returns:
    history: Dictionary containing loss and accuracy for both training and validation.
    """
    
    # Dictionary to store training & validation loss and accuracy over epochs
    history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
    
    for epoch in range(epochs):  # Loop over the number of epochs
        model.train()  # Set model to training mode
        total_loss, correct = 0, 0  # Initialize total loss and correct predictions
        
        # Training loop
        for inputs, labels in train_loader:
            optimizer.zero_grad()  # Reset gradients before each batch
            outputs = model(inputs).squeeze()  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            loss.backward()  # Backpropagation (compute gradients)
            optimizer.step()  # Update model parameters
            
            total_loss += loss.item()  # Accumulate batch loss
            correct += ((outputs > 0.5) == labels).sum().item()  # Count correct predictions
        
        # Compute average loss and accuracy for training
        train_loss = total_loss / len(train_loader)
        train_acc = correct / len(train_loader.dataset)
        
        # Validation phase (without gradient computation)
        model.eval()  # Set model to evaluation mode
        val_loss, val_correct = 0, 0

        all_preds = []
        all_labels = []

        with torch.no_grad():  # No need to compute gradients during validation
            for inputs, labels in val_loader:
                outputs = model(inputs).squeeze()  # Forward pass
                loss = criterion(outputs, labels)  # Compute loss
                val_loss += loss.item()  # Accumulate validation loss

                probs = torch.sigmoid(outputs)      # IMPORTANT si BCEWithLogitsLoss
                # preds = (probs > 0.5).int()
                preds = (probs > 0.35).float()

                # val_correct += ((outputs > 0.5) == labels).sum().item()  # Count correct predictions
                val_correct += (preds == labels).sum().item()

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        
        # Compute average loss and accuracy for validation
        val_loss /= len(val_loader)
        val_acc = val_correct / len(val_loader.dataset)
        
        # Store metrics in history dictionary
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['accuracy'].append(train_acc)
        history['val_accuracy'].append(val_acc)
        
        # Print training progress
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        print(confusion_matrix(all_labels, all_preds))
        print(classification_report(all_labels, all_preds))
    
    return history  # Return training history

history = train(model,
                train_loader=train_loader,
                val_loader=val_loader,
                criterion=criterion,
                optimizer=optimizer,
                epochs=10)

Epoch [1/10], Loss: 1.1697, Acc: 0.8656, Val Loss: 1.0971, Val Acc: 0.1322
[[   0 1451]
 [   0  221]]
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00      1451
         1.0       0.13      1.00      0.23       221

    accuracy                           0.13      1672
   macro avg       0.07      0.50      0.12      1672
weighted avg       0.02      0.13      0.03      1672



c:\Users\dubos\miniforge3\envs\Torche\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\Users\dubos\miniforge3\envs\Torche\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

c:\Users\dubos\miniforge3\envs\Torche\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



Epoch [2/10], Loss: 1.0253, Acc: 0.9208, Val Loss: 0.9159, Val Acc: 0.1519
[[  33 1418]
 [   0  221]]
              precision    recall  f1-score   support

         0.0       1.00      0.02      0.04      1451
         1.0       0.13      1.00      0.24       221

    accuracy                           0.15      1672
   macro avg       0.57      0.51      0.14      1672
weighted avg       0.89      0.15      0.07      1672

Epoch [3/10], Loss: 0.8146, Acc: 0.9674, Val Loss: 0.7028, Val Acc: 0.3355
[[ 340 1111]
 [   0  221]]
              precision    recall  f1-score   support

         0.0       1.00      0.23      0.38      1451
         1.0       0.17      1.00      0.28       221

    accuracy                           0.34      1672
   macro avg       0.58      0.62      0.33      1672
weighted avg       0.89      0.34      0.37      1672

Epoch [4/10], Loss: 0.6134, Acc: 0.9797, Val Loss: 0.5346, Val Acc: 0.6065
[[795 656]
 [  2 219]]
              precision    recall  f1-score 